# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [4]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# İngilizce stopwords
stop_words = set(stopwords.words('english'))

# Lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    # Küçük harfe çevir
    text = text.lower()

    # Sayıları kaldır
    text = re.sub(r'\d+', '', text)

    # Noktalama işaretlerini kaldır
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Kelimelere ayır
    words = text.split()

    # Stopwords kaldır
    words = [word for word in words if word not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # Tekrar metne dönüştür
    return ' '.join(words)

# Tüm metinleri temizle
data['clean_text'] = data['text'].fillna('').apply(preprocessing)

# Sonuçları kontrol et
data[['text', 'clean_text']].head()

,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gldcunixbcccolumbiaedu gary l dare subject sta...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,minerkuhubccukansedu subject ancient book orga...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivovsuperiorcarletonca vladimir zhivov subj...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Metinleri sayısal verilere dönüştür
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(data['clean_text'])

# LDA modelini oluştur
lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

# Modeli eğit
lda_model.fit(X)

# Sonuçları kontrol et
print("BoW matris boyutu:", X.shape)
print("LDA konu sayısı:", lda_model.n_components)
print("LDA components boyutu:", lda_model.components_.shape)

BoW matris boyutu: (1199, 19397)
LDA konu sayısı: 5
LDA components boyutu: (5, 19397)


##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [6]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [8]:
print_topics(lda_model, vectorizer)

Topic 0:
[('god', 221.7296907203389), ('would', 206.0742851758781), ('one', 197.965660529211), ('christ', 143.46837246230447), ('sin', 138.77948692099895), ('subject', 137.82830206179167), ('line', 131.84794029479113), ('organization', 124.77163849057169), ('also', 121.6473546177965), ('church', 111.1821438621761)]
Topic 1:
[('god', 1268.666650558696), ('christian', 718.4946687618447), ('one', 668.7037976963097), ('would', 642.2891342149403), ('people', 606.6187405823822), ('jesus', 517.8217419581895), ('subject', 512.8298681742281), ('say', 502.94829297008306), ('line', 477.55579390199637), ('know', 425.67255083988994)]
Topic 2:
[('pt', 182.63282938870796), ('la', 102.85599233006789), ('pitt', 30.767091497326344), ('scroll', 27.114142048631297), ('religious', 19.73511704469265), ('library', 18.020547363583148), ('sea', 17.934525390453288), ('animal', 17.436579351983024), ('jewish', 15.385237810754818), ('wash', 14.636257861874997)]
Topic 3:
[('team', 381.60529684459897), ('game', 317.

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [ ]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [9]:
# Yeni metni temizle
clean_example = [preprocessing(text) for text in example]

# Eğitilmiş vectorizer ile sayısallaştır
example_vectorized = vectorizer.transform(clean_example)

# LDA ile konu olasılıklarını tahmin et
topic_distribution = lda_model.transform(example_vectorized)

# Sonuçları görüntüle
print("Konu olasılıkları:", topic_distribution)

# En yüksek olasılıklı konuyu bul
predicted_topic = topic_distribution.argmax(axis=1)[0]

print("Tahmin edilen konu:", predicted_topic)
print("Konu olasılığı:", round(topic_distribution[0][predicted_topic], 3))

NameError: name 'example' is not defined

🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!